El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

# Preparación de datos

## Preprocesamiento y exploración de datos

### Inicialización

In [1]:
import warnings

import sys
import os
# Le dice python que busque liberrías ahí también
sys.path.append(os.path.join('src'))
import funciones_personales as fp

import math
import numpy as np
import pandas as pd

import seaborn as sns

import sklearn.linear_model
import sklearn.metrics
import sklearn.neighbors
import sklearn.preprocessing

from sklearn.model_selection import train_test_split

from IPython.display import display

from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")

/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.7/site-packages/sklearn/linear_model/least_angle.py:30: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  method='lar', copy_X=True, eps=np.finfo(np.float).eps,
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.7/site-packages/sklearn/linear_model/least_angle.py:167: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdo

In [ ]:
df= pd.read_csv('/datasets/car_data.csv')
print(df.info())
#crea la lista de filas con valores nan del dataframe
los_nanes= fp.mostrar_nan(df, mostrar=True)

## Búsqueda y tratamiento de valores nulos

**Se tienen un total de 108,555 filas con valores NaN de un total de 354,369 (~30.63 % del total de filas)**

* `VehicleType`: 37490
* `Gearbox`: 19833
* `Model`: 19705
* `FuelType`: 32895
* `NotRepaired`: 71154

¿ Qué tipo de datos tiene cada columna? y qué valores únicos tiene?

In [4]:
filas_c_nan= df[['VehicleType', 'Gearbox', 'Model', 'FuelType', 'NotRepaired']]

print(filas_c_nan.nunique())

for i in filas_c_nan.columns:
    print(f"Valores únicos en {i}: {filas_c_nan[i].unique()}")
    print('*'*50)

VehicleType      8
Gearbox          2
Model          250
FuelType         7
NotRepaired      2
dtype: int64
Valores únicos en VehicleType: [nan 'coupe' 'suv' 'small' 'sedan' 'convertible' 'bus' 'wagon' 'other']
**************************************************
Valores únicos en Gearbox: ['manual' 'auto' nan]
**************************************************
Valores únicos en Model: ['golf' nan 'grand' 'fabia' '3er' '2_reihe' 'other' 'c_max' '3_reihe'
 'passat' 'navara' 'ka' 'polo' 'twingo' 'a_klasse' 'scirocco' '5er'
 'meriva' 'arosa' 'c4' 'civic' 'transporter' 'punto' 'e_klasse' 'clio'
 'kadett' 'kangoo' 'corsa' 'one' 'fortwo' '1er' 'b_klasse' 'signum'
 'astra' 'a8' 'jetta' 'fiesta' 'c_klasse' 'micra' 'vito' 'sprinter' '156'
 'escort' 'forester' 'xc_reihe' 'scenic' 'a4' 'a1' 'insignia' 'combo'
 'focus' 'tt' 'a6' 'jazz' 'omega' 'slk' '7er' '80' '147' '100' 'z_reihe'
 'sportage' 'sorento' 'v40' 'ibiza' 'mustang' 'eos' 'touran' 'getz' 'a3'
 'almera' 'megane' 'lupo' 'r19' 'zafira' 'cadd

## Búaqueda y tratamiento de valores atípicos

In [ ]:
# Muestra los datos estadísticos de cada columna del df
for i in df.columns:
    print(f"Valores estadísticos en {i}: {df[i].describe()}")
    print('*'*50)

Valores estadísticos en DateCrawled: count               354369
unique               15470
top       05/03/2016 14:25
freq                    66
Name: DateCrawled, dtype: object
**************************************************
Valores estadísticos en Price: count    354369.000000
mean       4416.656776
std        4514.158514
min           0.000000
25%        1050.000000
50%        2700.000000
75%        6400.000000
max       20000.000000
Name: Price, dtype: float64
**************************************************
Valores estadísticos en VehicleType: count     316879
unique         8
top        sedan
freq       91457
Name: VehicleType, dtype: object
**************************************************
Valores estadísticos en RegistrationYear: count    354369.000000
mean       2004.234448
std          90.227958
min        1000.000000
25%        1999.000000
50%        2003.000000
75%        2008.000000
max        9999.000000
Name: RegistrationYear, dtype: float64
***********************

In [ ]:
# Se comentó ya que solo es una comprobación de datos**
'''
# Revisión de valores para los años de registro vehiculares
print(df['RegistrationYear'][(df['RegistrationYear'] < 1950)].value_counts().sort_index())
print('*'*50)
print(df['RegistrationYear'][(df['RegistrationYear'] > 2016)].value_counts().sort_index())
'''

1000     37
1001      1
1039      1
1111      3
1200      1
1234      4
1253      1
1255      1
1300      2
1400      1
1500      5
1600      2
1602      1
1688      1
1800      5
1910    101
1915      1
1919      1
1920      1
1923      2
1925      1
1927      1
1928      2
1929      7
1930      3
1931      1
1932      3
1933      3
1934      3
1935      4
1936      3
1937     11
1938      8
1940      2
1941      2
1942      3
1943      4
1944      2
1945      4
1946      1
1947      2
1948      3
1949      1
Name: RegistrationYear, dtype: int64
**************************************************
2017    10441
2018     3959
2019       25
2066        1
2200        1
2222        2
2290        1
2500        4
2800        2
2900        1
3000        7
3200        1
3500        1
3700        1
3800        1
4000        3
4100        1
4500        2
4800        1
5000       17
5300        1
5555        2
5600        1
5900        1
5911        2
6000        5
6500        1
7000        4
7100

**Valores Nulos:**
* Se decide rellenar los valores nulos con la palabra 'unknown'

**valores extraños en 'RegistrationYear'**
* Los primeros registros vehiculares datan de Francia en 1893.
* Existen 101 vehículos registrados en 1910. previo a este año hay registros de 1000 a 1800 (Deben modificarse).
* El año 2019 cuenta con 25 vehiculos registrados; depues los registros van hasta el año 2066 (deben modificarse).
    * Los años que se tomarán como verdaderos para este estudio son de 1910 a 2019.
    * Los valores que no entren en este rango se cambiaran por NAN para no afectar promedios.

**Valores = 0 en 'Power'**
* 40,225 de 354,369 (~11.35 del total)
    * Media: ~ 110.09
    * Mediana: ~ 105
    * El máximo (20,000: altísimo para auto de calle) y la desviación estandar (~ 189.85: elevada) apuntan a que los valores atípicos (outliers) están "jalando" hacia arriba la media.
* Se decide sustituir los valores 0 por el valor de la mediana (que es un poco más bajo que el de la media)

**Valores atípicos en 'Price'**
* Según ivestigaciones un precio muy bajo de un auto que enciende y rueda es de 500 dólares / euros
* 36,054 de 354,369 (~ 10.17% del total) son menores de 500
* Se decide eliminar ese ~10.17% de datos quedando con alrededor de 312,000 para usarlos en el entrenamiento del modelo. (siguen siendo una buena cantidad)
* De tratar de rellenar estos datos puede "confundir" al modelo ya que está usando como target datos "inventados", no reales pudiendo sesgar las predicciones.

**Columnas no necesarias para el modelo**
* `DateCrawled`
* `DateCreated`
* `NumberOfPictures` 
    * Podría afectar pero todos sus valores son 0, por lo que no afecta en este caso
* `PostalCode`
    * Puede reflejar el poder adquisitivo de una región pero no directamente al precio del vehículo.
    * Se decide eliminar para simplificar el modelo (en caso de requerirse podría tomarse en cuenta para futuros modelos)
* `LastSeen`

In [ ]:
# Rellena los nan con 'unknown' para cada columna con NAN
df = df.fillna({'VehicleType': 'unknown', 'Gearbox': 'unknown', 'Model': 'unknown', 'FuelType': 'unknown', 'NotRepaired': 'unknown'})

# Sustituye los valores de años fuera de rango por NAN
filtro_registro = ~df['RegistrationYear'].between(1910, 2019)
df.loc[filtro_registro, 'RegistrationYear'] = np.nan

# Cambia los valores de power de 0 por la mediana
df.loc[df['Power']== 0, 'Power'] = df['Power'].median()

# Elimina las filas cuyos datos en Price son menores de 500
df= df[df['Price'] >= 500]

# COMPROBACIONES
'''
# comprueba que los NAN se hayan sustituido por unknown
prueba_sin_nan= fp.mostrar_nan(df, mostrar=True)

# Comprueba que no queden valores para años fuera de rango
print(df['RegistrationYear'][(df['RegistrationYear'] < 1910)].value_counts().sort_index())
print('*'*50)
print(df['RegistrationYear'][(df['RegistrationYear'] > 2019)].value_counts().sort_index())

# Comprueba que no queden valores de 0 en power:
print(len(df[df['Power'] == 0]))

# Comprueba que no queden filas con price= 0
print(len(df[df['Price'] < 500]))
'''


# Elimina las columnas que no se utilizarán en el modelo renombrando al df limpio como df_clean
df_clean = df.drop(['DateCrawled', 'DateCreated', 'PostalCode', 'LastSeen', 'NumberOfPictures'], axis=1)

#Comprpbación para df_clean
'''
print(df_clean.columns)
print(df_clean.info())
'''


Total de filas con NaN: 0

Cantidad de NaN por columna:
  DateCrawled: 0
  Price: 0
  VehicleType: 0
  RegistrationYear: 0
  Gearbox: 0
  Power: 0
  Model: 0
  Mileage: 0
  RegistrationMonth: 0
  FuelType: 0
  Brand: 0
  NotRepaired: 0
  DateCreated: 0
  NumberOfPictures: 0
  PostalCode: 0
  LastSeen: 0



## Se requeriran tres tratamientos diferentes del dataframe df_clean para probar los modelos

* `df_clean`: Dataframe con los targets y features limpios (sin nan) sin escalar ni codificar.
    * Se utilizará para los modelos: **CatBoost** y **LightGBM**
* `df_hotcode_scaled`: Dataframe con los targets y features limpios, escalados y codificados.
    * Se utilizará para el modelo: **Regresión Lineal** (modelo dummy)
* `df_hotcode`: Dataframe con los target y features limpios, codificados sin escalar
    * Se utilizará para el modelo: **RandomForest** y **XGBoost**

In [ ]:
# lista de las columnas categóricas y numéricas de los df
categoric_cols= df_clean.select_dtypes(include=['object']).columns
numeric_cols= ['RegistrationYear', 'Power', 'Mileage', 'RegistrationMonth']

# Modelos Dummy, Regresión lineal

## Separación de datos:

* df_clean cuenta con 318,315 filas y 11 columnas
    * Para aminorar tiempos de computo se decide dividir datos en train, validation y test. De esta manera quedarían aproximadamente:
        * ~222,820 para entrenar (datos suficientes)
        * ~47,747 para validar (ajuste de hiperparámetros)
        * ~47,747 para realizar evaluación final

## orden de los pasos

1. Separar features (X) y target (y).
2. Dividir en train/validations/test (70/15/15) con `train_test_split`.
3. Aplicar One-Hot Encoding a las columnas categóricas.
4. Escalar las columnas numéricas

Obligatorios:

    📏 Regresión Lineal → prueba de cordura (sanity check)
    🌲 Random Forest → con ajuste de hiperparámetros
    ⚡ LightGBM → con ajuste de hiperparámetros

Opcionales:

    🐱 CatBoost
    🚀 XGBoost

Y la métrica para evaluar todos será RMSE.

In [ ]:
# 1 Separación de target y features
x_reglin = df_clean.drop('Price', axis=1)
y_reglin = df_clean['Price']

## Separación de datos


In [ ]:
# Crea los conjuntos features y target
df_features = df_clean.drop('Price', axis=1)
df_target = df_clean['Price']

# Primera división: 85% (train+test) vs 15% (validation)
features_all, features_valid, target_all, target_valid = train_test_split(
    df_features, df_target, 
    test_size=0.15, 
    random_state=54321
)
# Segunda división: 70% train vs 15% test (del 85% restante)
# 15/85 ≈ 0.176 para obtener 15% del total
features_train, features_test, target_train, target_test = train_test_split(
    features_all, target_all, 
    test_size=0.176,  # 15/85 = 0.176
    random_state=54321
)

Index(['Price', 'VehicleType', 'RegistrationYear', 'Gearbox', 'Power', 'Model',
       'Mileage', 'RegistrationMonth', 'FuelType', 'Brand', 'NotRepaired',
       'NumberOfPictures'],
      dtype='object')


In [8]:
print(los_nanes.shape)
nan_rowslos_nanes.sample(10, random_state=2010))
print(df.shape)
print((df.shape[0]/los_nanes.shape[0])*100)

SyntaxError: invalid syntax (1110741675.py, line 2)

In [ ]:
df.describe()

,Price,RegistrationYear,Power,Mileage,RegistrationMonth,NumberOfPictures,PostalCode
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000,354369.0,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645,0.0,50508.689087
std,4514.158514,90.227958,189.850405,37905.341530,3.726421,0.0,25783.096248
min,0.000000,1000.000000,0.000000,5000.000000,0.000000,0.0,1067.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000,0.0,30165.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000,0.0,49413.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000,0.0,71083.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000,0.0,99998.000000


In [ ]:
df.nunique()

DateCrawled          15470
Price                 3731
VehicleType              8
RegistrationYear       151
Gearbox                  2
Power                  712
Model                  250
Mileage                 13
RegistrationMonth       13
FuelType                 7
Brand                   40
NotRepaired              2
DateCreated            109
NumberOfPictures         1
PostalCode            8143
LastSeen             18592
dtype: int64

In [ ]:
print(df['RegistrationMonth'].unique())
num= df['RegistrationMonth'][df['RegistrationMonth'] == 0].count()
print((num/df.shape[0])*100)
print(df.shape)
print(num)

[ 0  5  8  6  7 10 12 11  2  3  1  4  9]
10.540425375808832
(354369, 16)
37352


In [ ]:
ceros= df[df['RegistrationMonth'] == 0]
ceros.describe()

,Price,RegistrationYear,Power,Mileage,RegistrationMonth,NumberOfPictures,PostalCode
count,37352.000000,37352.000000,37352.000000,37352.000000,37352.0,37352.0,37352.000000
mean,1907.968757,2010.342927,70.120957,127888.466481,0.0,0.0,47556.917916
std,2796.876820,252.543228,237.755857,44678.059214,0.0,0.0,26001.888358
min,0.000000,1000.000000,0.000000,5000.000000,0.0,0.0,1067.000000
25%,300.000000,1997.000000,0.000000,125000.000000,0.0,0.0,26802.000000
50%,900.000000,2000.000000,60.000000,150000.000000,0.0,0.0,45966.000000
75%,2250.000000,2006.000000,110.000000,150000.000000,0.0,0.0,66482.000000
max,20000.000000,9999.000000,19211.000000,150000.000000,0.0,0.0,99998.000000


In [ ]:
# Cuenta los valores nulos de columna price
print(df['Price'].isnull().sum())

0


**Hay precios de 0!!!!**
* ¿cuántos son?
* Tienen algo más en común?
**Hay 13 valores para 12 meses.**
* La cantidad de filas en las que está 0 como el mes de registro corresponde a 37,532. (10.54%)  ¿Qué significa el 0?. ¿cuantos hay?
**`NimberOfPictures` siempre es 0. ¿Qué hacemos con eso?**

## Entrenamiento del modelo 

## Análisis del modelo

# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [ ]  El código no tiene errores- [ ]  Las celdas con el código han sido colocadas en orden de ejecución- [ ]  Los datos han sido descargados y preparados- [ ]  Los modelos han sido entrenados
- [ ]  Se realizó el análisis de velocidad y calidad de los modelos